# Thoracic Disease Detection — Reproducible Colab Run

This notebook downloads NIH ChestXray14 to Colab's local SSD, audits all 112,120 images, trains DenseNet-121 and ViT-B/16, evaluates the best checkpoints, generates DenseNet Grad-CAM explanations, and produces the final architecture comparison.

Use an A100 GPU with High-RAM when available. Model checkpoints and evaluations are written to Google Drive. Dataset files remain on the temporary Colab disk.

> Research software only. These models are not intended for diagnosis or clinical use.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Repository and environment

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sethu-ram-reddy/thoracic-disease-detection.git"
REPO_DIR = Path("/content/thoracic-disease-detection")

if REPO_DIR.exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())

In [ ]:
import importlib

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO_DIR)],
    check=True,
)

SOURCE_DIR = REPO_DIR / "src"
source_path = str(SOURCE_DIR)
if source_path not in sys.path:
    sys.path.insert(0, source_path)

existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    source_path
    if not existing_pythonpath
    else source_path + os.pathsep + existing_pythonpath
)

importlib.invalidate_caches()
thoracic_ai = importlib.import_module("thoracic_ai")

print("Package imported from:", thoracic_ai.__file__)

## Download and audit NIH ChestXray14

The Kaggle mirror is downloaded directly to Colab's local SSD. The extracted dataset is approximately 42 GB, so confirm that the runtime has sufficient disk space.

In [ ]:
import kagglehub

LOCAL_DATA_ROOT = "/content/data/nih-chest-xray-kaggle"
downloaded_path = kagglehub.dataset_download(
    "nih-chest-xrays/data",
    output_dir=LOCAL_DATA_ROOT,
)
LOCAL_DATA_DIR = Path(downloaded_path).resolve()
png_count = sum(1 for _ in LOCAL_DATA_DIR.rglob("*.png"))

print("Dataset directory:", LOCAL_DATA_DIR)
print("PNG images found:", png_count)
if png_count != 112120:
    raise RuntimeError(f"Expected 112120 PNG images, found {png_count}.")

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/audit_dataset.py",
        "--data-root",
        str(LOCAL_DATA_DIR),
    ],
    check=True,
)

## DenseNet-121

Training is skipped when a completed `best.pt` already exists in Drive. Remove the output directory if a clean rerun is required.

In [ ]:
DENSENET_DIR = Path(
    "/content/drive/MyDrive/Thoracic_Disease_Detection/outputs/densenet121"
)

if (DENSENET_DIR / "best.pt").exists():
    print("DenseNet checkpoint already exists; training skipped.")
else:
    subprocess.run(
        [
            sys.executable,
            "scripts/train.py",
            "--config",
            "configs/densenet121.yaml",
            "--data-root",
            str(LOCAL_DATA_DIR),
        ],
        check=True,
    )

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/evaluate.py",
        "--config",
        "configs/densenet121.yaml",
        "--data-root",
        str(LOCAL_DATA_DIR),
    ],
    check=True,
)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/generate_gradcam.py",
        "--config",
        "configs/densenet121.yaml",
        "--data-root",
        str(LOCAL_DATA_DIR),
        "--samples",
        "12",
    ],
    check=True,
)

## ViT-B/16

In [ ]:
VIT_DIR = Path(
    "/content/drive/MyDrive/Thoracic_Disease_Detection/outputs/vit_b16"
)

if (VIT_DIR / "best.pt").exists():
    print("ViT checkpoint already exists; training skipped.")
else:
    subprocess.run(
        [
            sys.executable,
            "scripts/train.py",
            "--config",
            "configs/vit_b16.yaml",
            "--data-root",
            str(LOCAL_DATA_DIR),
        ],
        check=True,
    )

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/evaluate.py",
        "--config",
        "configs/vit_b16.yaml",
        "--data-root",
        str(LOCAL_DATA_DIR),
    ],
    check=True,
)

## Compare the completed experiments

In [ ]:
COMPARISON_DIR = Path(
    "/content/drive/MyDrive/Thoracic_Disease_Detection/outputs/comparison"
)

subprocess.run(
    [
        sys.executable,
        "scripts/compare_models.py",
        "--densenet-dir",
        str(DENSENET_DIR),
        "--vit-dir",
        str(VIT_DIR),
        "--output-dir",
        str(COMPARISON_DIR),
    ],
    check=True,
)

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

model_summary = pd.read_csv(COMPARISON_DIR / "model_summary.csv")
disease_comparison = pd.read_csv(
    COMPARISON_DIR / "disease_level_comparison.csv"
)
display(model_summary.round(4))
display(disease_comparison.round(4))

comparison_image = plt.imread(COMPARISON_DIR / "model_comparison.png")
plt.figure(figsize=(10, 6))
plt.imshow(comparison_image)
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
GRADCAM_DIR = DENSENET_DIR / "evaluation/gradcam"
gradcam_files = sorted(GRADCAM_DIR.glob("*.png"))
columns = 2
rows = math.ceil(len(gradcam_files) / columns)
figure, axes = plt.subplots(rows, columns, figsize=(18, 4.5 * rows))
axes = axes.flatten()

for axis, image_path in zip(axes, gradcam_files, strict=False):
    axis.imshow(plt.imread(image_path))
    axis.set_title(image_path.stem, fontsize=10)
    axis.axis("off")

for axis in axes[len(gradcam_files):]:
    axis.axis("off")

figure.tight_layout()
plt.show()

## Completed outputs

The Drive output tree now contains the best and latest checkpoints, resolved configurations, patient-safe split manifests, training histories, validation-selected thresholds, test predictions, metric tables, comparison figures, and Grad-CAM examples.